# Lesson 4: 👀 QC flags 

### 🎯 Learning Objectives
In this lesson, we will visualize the float time series downloaded in Lesson 3 and understand their spatial and temporal coverage. Specifically, we will examine:

* the time of sampling (was the profiling done at different time of the day for each cycle?)
* the frequency of sampling (how many days are apart between profiling cycles?)
* the vertical extent and resolution of the raw data (how deep and frequent are the vertical profiles available?)
* the availability of the raw data (are there missing values in the time series?)

We will also learn about the quality-control (QC) flags assigned to each sample of every profile in the time series. We will learn how to use the QC flags to filter the time series to retain only good values, while excluding suspicious values.

### 🛠️ Prerequisites
Before starting this lesson, make sure that you have completed **Lesson 3**.

### ❓ How to Use This Notebook
* 📚 **Read** the tutorial text blocks carefully, as they provide the essential background information behind the code.
* ▶️ **Run** each code cell sequentially by clicking the cell and pressing `Shift + Enter`.
* 📝 **Exercise** your knowledge! At the end of this notebook, we provide active learning exercises where you will need to write the code yourself.

### 🚀 Ready? Let's Get Started!
---

## 📚 Tutorials

### Import libraries

▶️ Run the cell below to import relevant Python libraries for this lesson.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

### Load the float time series

▶️ Run the cell below to load the float time series downloaded in Lesson 3.

In [ ]:
wmoid = 5906513

in_file = f"../floats/{wmoid}/{wmoid}_Sprof.nc"
ds = xr.open_dataset(in_file) 
ds

### Take an initial look at the float time series

☝️ We see from the output that there are multiple dimensions in the float time series. The most relevant ones are:

* **N_PROF**: the number of profiles. In this example, the time series consist of 123 vertical profiles.
* **N_LEVELS**: the number of depth levels. In this example, there are 554 vertical levels in each profile. However, note that each profile has its own pressure values and not necessarily has values at all levels (i.e., there are missing values). We will visualize this shortly in this lesson.

Next, click on **Data variables** above to see the list of available variables in the time series. There are so many (93 in this example)! Fortunately, we only use a few of these.

The following variables are 1D arrays (N_PROF) containing each profile's sampling date and location:

* **JULD**: date and time
* **LATITUDE**: latitude (-90 to 90)
* **LONGITUDE**: longitude (-180 to 180)

The following variables are 2D arrays (N_PROF x N_LEVELS) containing each profile's physical oceanoraphic data:

* **PRES_ADJUSTED**: adjusted pressure (dbar)
* **TEMP_ADJUSTED**: adjusted temperature (deg-c)
* **TEMP_ADJUSTED_QC**: quality-control flag for adjusted temperature (-)
* **PSAL_ADJUSTED**: adjusted salinity (psu)
* **PSAL_ADJUSTED_QC**: quality-control flag for adjusted salinity (-)

Note that we will use adjusted values (e.g., **PRES_ADJUSTED**) instead of raw values (e.g., **PRES**) because the adjusted values are always better.

### QC flags

Quality-control (QC) flags (i.e., **\*_ADJUSTED_QC** in our case) refer to the quality of the data values. A standard practice is to only use data with QC flags of 1, 2, 5, or 8. [The Argo Online School](https://euroargodev.github.io/argoonlineschool/Lessons/L02_TheArgoData/Chapter24_QualityFlagMean.html) provides a nice summary of QC flags for Argo, including what each flag means.

The description for the biogeochemical variables (2D arrays) is omitted here because it was already covered in Lesson 1 (see **Filter by BGC parameters**). Similar to pressure, temperature, and salinity, we will use adjusted values (**\*_ADJUSTED**) and corresponding QC flags (**\*_ADJUSTED_QC**).

We will not consider the QC flags assigned to pressure (**PRES_ADJUSTED_QC**). We assume that filtering by the QC flags assigned to each variable is adequate. 

We will start with visualizing the sampling times and cycles to understand whether this float was profiling at a specific time of the day or at various times of the day AND whether the profiling was done at a constant interval or were there gaps in temporal coverage. We have some understanding regarding the sampling frequency from the timeline map in Lesson 2, but here we will look into this more in detail for a specific float.

### The function `time_dt`

▶️ Run the cell below to the function to calculate and visualize the sampling time and cycle

In [ ]:
def time_and_dt(ds):

    """
    This function extracts the sampling times and calculates the profiling cycles
    and visualizes them

    INPUT:
    * ds: the xarray dataset of the float time series

    OUTPUT:
    * utc_hours: sampling times in hours and in the UTC time zone
                 (associated figure: ../figures/sampling_time_{wmoid}.png)
    * dt_cycle: the profile cycles in days
                (associated figure: ../figures/sampling_cycle_{wmoid}.png)
    """

    # Obtain the WMO
    wmoid = str(ds["PLATFORM_NUMBER"].astype(int).values[0])

    # sampling time
    plt.figure()
    utc_hours = ds["JULD"].dt.hour + (ds["JULD"].dt.minute / 60) + (ds["JULD"].dt.second / 3600)
    utc_hours.plot(marker="|",ls="None")
    plt.title(f"Sampling time (WMO ID: {wmoid})")
    plt.ylim(0,24)
    plt.ylabel('UTC (hours)')
    plt.tight_layout()
    
    # sampling cycle
    plt.figure()
    dt_cycle = ds["JULD"].diff(dim='N_PROF') / np.timedelta64(1, 'D')
    dt_cycle.plot(marker="|",ls="None")
    plt.title(f"Sampling cycle (WMO ID: {wmoid})")
    plt.ylabel(r'$\Delta$t (days)')
    plt.tight_layout()
    
    return utc_hours, dt_cycle

▶️ Run the cell below to use the function

In [ ]:
utc_hours, dt_cycle = time_and_dt(ds)

☝️ Do you see two figures? The first one shows the time in which each profiling took place. The time is given in UTC. We can see that the sampling time for this float is well distributed. This is a standard practice for a float that wants to avoid the sampling bias due to diurnal variations.

The second figure shows the time difference between consecutive profiles. We see that the profiling was done pretty consistently at an interval of about 10 days (which is also a standard frequency for Argo) until the last few cycles in which the intervals exceeded 40 days. For scientific analysis, we probably want to exclude these profiles due to inconsistent intervals.

### The function `filter_qc`

Next, we will filter the data by QC flags.

▶️ Run the cell below to define a function that filters the data by the selected QC flags.

In [ ]:
def filter_qc(ds_in, name_in, qc_in):
    """
    This function filters the data based on their QC flags.

    INPUT:
    * ds_in: xarray dataset of the float time series
    * name_in: name of the variable (string)
    * qc_in: list of the accepted QC flags (list)

    OUTPUT:
    * data_valid: QC-filtered adjusted values
    """    
    data = ds_in[name_in]
    qc = ds_in[f"{name_in}_QC"] 
    valid = qc.astype(str).isin([str(x) for x in qc_in])
    data_valid = data.where(valid, other=np.nan)
    
    return data_valid

### The function `plot_2d_heatmap`

Next we want to visualize the 2D arrays of the adjusted variables (pressure, temperature, salinity, and the six standard BGC parameters) and compare them without and with the QC filter.

▶️ Run the cell below to define a function that attempts to plot 2D heatmap (pcolormesh) for each variable. It returns a text if the variable is not available in the dataset or if all values are missing for that variable.

In [ ]:
def plot_2d_heatmap(ds_in, var_list_in, qc_in=None):
    """
    This function draws 2D heatmaps for the standard variables for BGC-Argo floats listed below (vars_original).

    INPUT:
    * ds_in: xarray dataset

    OUTPUT:
    * 2D heatmaps (we do not save these as images as this is for diagnostics only).
    """

    # If input is a string, make it a list
    if isinstance(var_list_in, str):
        var_list_in = [var_list_in]
        
    for j in range(len(var_list_in)): # loop over variables
        try:
            var_list_in[j] in ds_in.data_vars

            # Filter by QC
            if qc_in:
                ds_good = filter_qc(ds_in, var_list_in[j], qc_in)
            # Do not filter by QC
            else:
                ds_good = ds_in[var_list_in[j]].copy()
            
            if ds_good.notnull().any(): # if finite values exit
                plt.figure()
                mmin = ds_good.min().astype(np.float32).values
                mmax = ds_good.max().astype(np.float32).values
                ds_good.plot(x="N_PROF",vmin=mmin,vmax=mmax)
                plt.title(f"min: {mmin}, max: {mmax}")            
                plt.gca().invert_yaxis()
                plt.tight_layout()
            else:
              print(f'No VALID value available from {var_list_in[j]}')  
        except:
            print(f'No variable named {var_list_in[j]} available')

▶️ Run the cell below to define the variable list and then run the function to plot the adjusted data without the QC filter.

In [ ]:
var_list = ['PRES_ADJUSTED',
            'TEMP_ADJUSTED',
            'PSAL_ADJUSTED',
            'DOWNWELLING_PAR_ADJUSTED',
            'NITRATE_ADJUSTED',
            'CHLA_ADJUSTED',
            'BBP700_ADJUSTED',
            'DOXY_ADJUSTED',
            'PH_IN_SITU_TOTAL_ADJUSTED',
           ]

plot_2d_heatmap(ds, var_list)

☝️ Can you scroll down the heatmap to see if you produced several heatmaps? From these heatmaps, you may notice that:

* **PAR** is not available (as expected from the information extracted from the index file in Lesson 1).
* The vertical coverage is the upper 1,600 m (dbar).
* Compared to temperature and salinity, the vertical resolutions of the BGC parameters are coarse.
* **Nitrate** and **chlorophyll-a** have negative values. 

☝️ We have plotted all adjusted values, but in practice, we need to filter out bad-quality values using the QC flags.

▶️ **Run** the cell below to define the accepted QC flags (**1** for this example, which means **good**) and plot the heatmaps after applying the QC filter.

In [ ]:
qc_list = [1]

plot_2d_heatmap(ds, var_list, qc_list)

☝️ By comparing with the heatmaps without the QC filter, we can see that there is hardly any difference, meaning that this float time series are of good quality. In the exercises below, we will see cases where the QC filter helps to remove some *bad* profiles.

**This is the end of the tutorials for this lesson. Hope you enjoyed it!**

---

## 📝 Exercises

### Exercise 1: Visualize the time series for the float 5905531downloaded in Ex 2 of Lesson 3

1. Plot the sampling times and profiling cycles
1. Plot the 2D heatmaps of the adjusted values without the QC filter

📝 Write the code yourself below and ▶️ run it!

🎨 Next, plot the 2D heatmaps again but with the QC filter (1,2,5,8).

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

wmoid = 5905531
in_file = f"../floats/{wmoid}/{wmoid}_Sprof.nc"
ds = xr.open_dataset(in_file) 

plt.close('all')

time_and_dt(ds)

plot_2d_heatmap(ds, var_list, qc_in=None)

plot_2d_heatmap(ds, var_list, qc_in=[1,2,5,8])

### Exercise 2: Visualize the time series for the float 5906475 downloaded in Ex 3 of Lesson 3

1. Plot the sampling times and profiling cycles
1. Plot the 2D heatmaps of the adjusted values without the QC filter

📝 Write the code yourself below and ▶️ run it!

🎨 Next, plot the 2D heatmaps again but with the QC filter (1,2,5,8).

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

wmoid = 5906475
in_file = f"../floats/{wmoid}/{wmoid}_Sprof.nc"
ds = xr.open_dataset(in_file) 

plt.close('all')

time_and_dt(ds)

plot_2d_heatmap(ds, var_list, qc_in=None)

plot_2d_heatmap(ds, var_list, qc_in=[1,2,5,8])

### Exercise 3: Visualize the time series for the float 5906476 downloaded in Ex 3 of Lesson 3 (the sister float)

1. Plot the sampling times and profiling cycles
1. Plot the 2D heatmaps of the adjusted values without the QC filter

📝 Write the code yourself below and ▶️ run it!

🎨 Next, plot the 2D heatmaps again but with the QC filter (1,2,5,8). 

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

wmoid = 5906476
in_file = f"../floats/{wmoid}/{wmoid}_Sprof.nc"
ds = xr.open_dataset(in_file) 

plt.close('all')

time_and_dt(ds)

plot_2d_heatmap(ds, var_list, qc_in=None)

plot_2d_heatmap(ds, var_list, qc_in=[1,2,5,8])


---

This is the end of the lesson. If you are using **Binder**, don't forget to dowload the files created in this lesson before you lose connection!

Well done 🎉, take a break 💤, have another cup ☕, and move to the next lesson ✍️ when you are ready 💪

While your memory is fresh, please feel free to provide your user experience on this lesson by visiting [this link](https://forms.gle/oAGmz5RTW4Pp46bt7). Thanks!